In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
import numpy as np

def evaluate_nn_model(model, X_test, Y_actual):
    """
    Evaluate a NN model that outputs probabilities.
    Computes ROC, Precision-Recall curves, and best F1 score automatically.
    """
    
    # Predict probabilities
    Y_prob = model.predict(X_test).flatten()  # flatten in case output is (n,1)
    
    # --- Compute best F1 score by sweeping thresholds ---
    thresholds = np.linspace(0, 1, 101)  # thresholds from 0.0 to 1.0
    f1_scores = [f1_score(Y_actual, (Y_prob >= t).astype(int)) for t in thresholds]
    best_f1 = max(f1_scores)
    best_threshold = thresholds[np.argmax(f1_scores)]
    
    # Binary predictions using the best threshold
    Y_pred_best = (Y_prob >= best_threshold).astype(int)
    
    # --- Metrics ---
    metrics = {
        "F1": best_f1,
        "Best_Threshold": best_threshold,
        "AUC": roc_auc_score(Y_actual, Y_prob)
    }

    # --- ROC curve ---
    fpr, tpr, roc_thresholds = roc_curve(Y_actual, Y_prob)
    metrics["ROC"] = {"fpr": fpr, "tpr": tpr, "thresholds": roc_thresholds}

    # --- Precision-Recall curve ---
    precision, recall, pr_thresholds = precision_recall_curve(Y_actual, Y_prob)
    avg_precision = average_precision_score(Y_actual, Y_prob)
    metrics["Precision_Recall_Curve"] = {"precision": precision, "recall": recall, "thresholds": pr_thresholds}
    metrics["Average_Precision"] = avg_precision

    # --- Plotting ---
    plt.figure(figsize=(12,5))

    # ROC Curve
    plt.subplot(1,2,1)
    plt.plot(fpr, tpr, label=f"AUC = {metrics['AUC']:.2f}", color='blue', linewidth=2)
    plt.plot([0,1], [0,1], linestyle='--', color='gray')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend(loc='lower right')

    # Precision-Recall Curve
    plt.subplot(1,2,2)
    plt.plot(recall, precision, label=f"Average Precision = {avg_precision:.2f}", color='green', linewidth=2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend(loc='lower left')

    plt.tight_layout()
    plt.show()

    return metrics


Sup
